# 7. Dashboard Consumption & Streamlit Preparation

Το παρόν notebook αποτελεί το επόμενο στάδιο μετά το `06_Dashboard_Ready_Market_Outputs.ipynb`
και έχει ως στόχο να καταναλώσει τα ήδη παραγμένα dashboard-ready outputs του project,
να οργανώσει τις βασικές ενότητες προβολής του dashboard και να προετοιμάσει καθαρές,
τυποποιημένες και αναπαραγώγιμες εισόδους για μελλοντική ενσωμάτωση σε εφαρμογή Streamlit.

Σε αυτό το στάδιο δεν παράγεται νέα πρωτογενής ανάλυση της αγοράς.
Αντίθετα, δίνεται έμφαση στη σωστή φόρτωση, στον έλεγχο συνέπειας, στη δομημένη οργάνωση
των exported πινάκων και γραφημάτων, καθώς και στη δημιουργία component-ready δομών
για KPI cards, chart registries και dashboard sections.

## Μεθοδολογικό πλαίσιο και σχεδιαστική λογική

Η λειτουργία του παρόντος notebook βασίζεται στις ακόλουθες αρχές:

1. **Consumption layer και όχι νέο analytical stage**  
   Το notebook δεν επαναλαμβάνει την ανάλυση που έχει ήδη ολοκληρωθεί στα προηγούμενα στάδια,
   αλλά αξιοποιεί τα ήδη εξαχθέντα outputs του Notebook 6 ως επίσημες εισόδους για dashboard χρήση.

2. **Reproducible workflow**  
   Κάθε βήμα φόρτωσης, ελέγχου, μετασχηματισμού και εξαγωγής πρέπει να είναι πλήρως αναπαραγώγιμο,
   χωρίς χειροκίνητες παρεμβάσεις εκτός notebook.

3. **Σαφής διαχωρισμός μεταξύ data, metadata και presentation logic**  
   Θα γίνεται διάκριση ανάμεσα:
   - στα πραγματικά dashboard input tables,
   - στα registries που περιγράφουν sections, cards και charts,
   - και στα presentation-oriented στοιχεία που θα αξιοποιηθούν αργότερα σε Streamlit app.

4. **Καθαρή και επεκτάσιμη δομή κώδικα**  
   Ο κώδικας θα γραφτεί με μικρές βοηθητικές συναρτήσεις, σαφή ονοματοδοσία,
   ελεγχόμενα paths και validation checks, ώστε να παραμένει ευανάγνωστος και εύκολος στη συντήρηση.

5. **Προετοιμασία για μελλοντικό Streamlit integration**  
   Οι τελικές δομές που θα παραχθούν θα είναι σχεδιασμένες έτσι ώστε να μπορούν να χρησιμοποιηθούν
   εύκολα σε μελλοντική dashboard εφαρμογή, χωρίς να απαιτείται ανασχεδιασμός των βασικών inputs.

In [1]:
# ============================================
# Notebook 7 - Imports και βασικές ρυθμίσεις
# ============================================

from __future__ import annotations

import sys
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from IPython.display import display

# --------------------------------------------
# 1. Βασικές ρυθμίσεις notebook
# --------------------------------------------

warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_rows", 100)
pd.set_option("display.width", 140)
pd.set_option("display.float_format", "{:,.2f}".format)

sns.set_theme(style="whitegrid", context="notebook")

NOTEBOOK_ID = 7
NOTEBOOK_NAME = "Dashboard Consumption & Streamlit Preparation"
NOTEBOOK_EXPORT_PREFIX = "notebook7"

# --------------------------------------------
# 2. Έλεγχος έκδοσης Python
# --------------------------------------------

python_version = sys.version_info

if python_version < (3, 11):
    raise RuntimeError(
        "Το notebook απαιτεί Python 3.11+ για ασφαλή συμβατότητα με το τρέχον project."
    )

print(f"Notebook {NOTEBOOK_ID}: {NOTEBOOK_NAME}")
print(f"Python version: {sys.version.split()[0]}")
print("Το περιβάλλον ρυθμίστηκε επιτυχώς.")

Notebook 7: Dashboard Consumption & Streamlit Preparation
Python version: 3.13.1
Το περιβάλλον ρυθμίστηκε επιτυχώς.


In [2]:
# ============================================
# Notebook 7 - Paths και βοηθητικές συναρτήσεις
# ============================================

def find_project_root(start_path: Path) -> Path:
    """
    Εντοπίζει το root του project αναζητώντας βασικά αρχεία και φακέλους.

    Η συνάρτηση είναι χρήσιμη ώστε το notebook να μπορεί να τρέχει σωστά
    είτε από το root του repository είτε από τον φάκελο notebooks/.
    """
    current_path = start_path.resolve()

    for candidate_path in [current_path] + list(current_path.parents):
        has_data_dir = (candidate_path / "data").exists()
        has_notebooks_dir = (candidate_path / "notebooks").exists()
        has_readme_file = (candidate_path / "README.md").exists()

        if has_data_dir and has_notebooks_dir and has_readme_file:
            return candidate_path

    raise FileNotFoundError(
        "Δεν ήταν δυνατό να εντοπιστεί το root του project."
    )


def ensure_directory(path: Path) -> None:
    """
    Δημιουργεί τον φάκελο αν δεν υπάρχει ήδη.
    """
    path.mkdir(parents=True, exist_ok=True)


def read_csv_strict(file_path: Path, **kwargs) -> pd.DataFrame:
    """
    Διαβάζει CSV αρχείο και επιστρέφει DataFrame.

    Αν το αρχείο δεν υπάρχει, εμφανίζει καθαρό και επεξηγηματικό σφάλμα.
    """
    if not file_path.exists():
        raise FileNotFoundError(f"Δεν βρέθηκε το αρχείο: {file_path}")

    return pd.read_csv(file_path, **kwargs)


def validate_required_columns(
    df: pd.DataFrame,
    required_columns: list[str],
    table_name: str,
) -> None:
    """
    Ελέγχει αν ένα DataFrame περιέχει όλες τις απαιτούμενες στήλες.
    """
    missing_columns = [col for col in required_columns if col not in df.columns]

    if missing_columns:
        raise ValueError(
            f"Το table '{table_name}' δεν περιέχει τις απαιτούμενες στήλες: {missing_columns}"
        )


# --------------------------------------------
# 1. Εντοπισμός project root
# --------------------------------------------

PROJECT_ROOT = find_project_root(Path.cwd())

# --------------------------------------------
# 2. Βασικά paths του project
# --------------------------------------------

DATA_DIR = PROJECT_ROOT / "data"
RAW_DIR = DATA_DIR / "raw"
PROCESSED_DIR = DATA_DIR / "processed"

PLOTS_DIR = PROJECT_ROOT / "plots"
NOTEBOOKS_DIR = PROJECT_ROOT / "notebooks"
SRC_DIR = PROJECT_ROOT / "src"

# --------------------------------------------
# 3. Διασφάλιση βασικών φακέλων
# --------------------------------------------

ensure_directory(PROCESSED_DIR)
ensure_directory(PLOTS_DIR)

# --------------------------------------------
# 4. Σύντομος έλεγχος paths
# --------------------------------------------

print("Project root:", PROJECT_ROOT)
print("Processed data dir:", PROCESSED_DIR)
print("Plots dir:", PLOTS_DIR)
print("Notebooks dir:", NOTEBOOKS_DIR)
print("Src dir:", SRC_DIR)

Project root: C:\Users\athin\Desktop\Car-Market-Analysis-Attica
Processed data dir: C:\Users\athin\Desktop\Car-Market-Analysis-Attica\data\processed
Plots dir: C:\Users\athin\Desktop\Car-Market-Analysis-Attica\plots
Notebooks dir: C:\Users\athin\Desktop\Car-Market-Analysis-Attica\notebooks
Src dir: C:\Users\athin\Desktop\Car-Market-Analysis-Attica\src


## Φόρτωση των dashboard-ready outputs του Notebook 6

Στο παρόν βήμα θα φορτωθούν τα βασικά exported αρχεία του `06_Dashboard_Ready_Market_Outputs.ipynb`
από τους φακέλους `data/processed/` και `plots/`.

Ο στόχος δεν είναι μόνο η ανάγνωση των αρχείων, αλλά και η επιβεβαίωση ότι:
- όλα τα αναμενόμενα CSV summaries υπάρχουν,
- όλα τα απαιτούμενα plot assets είναι διαθέσιμα,
- οι εισροές του dashboard stage είναι πλήρεις και συνεπείς πριν προχωρήσουμε
  στη δημιουργία sections, cards, registries και streamlit-ready δομών.

In [3]:
# ============================================
# Notebook 7 - Inventory των outputs του Notebook 6
# ============================================

def build_file_inventory(directory: Path, pattern: str, asset_type: str) -> pd.DataFrame:
    """
    Δημιουργεί inventory πίνακα για τα αρχεία ενός φακέλου
    με βάση ένα glob pattern.
    """
    matched_files = sorted(directory.glob(pattern))

    records = []
    for file_path in matched_files:
        records.append(
            {
                "asset_type": asset_type,
                "file_name": file_path.name,
                "suffix": file_path.suffix.lower(),
                "parent_dir": file_path.parent.name,
                "absolute_path": str(file_path),
            }
        )

    return pd.DataFrame(records)


# --------------------------------------------
# 1. Εντοπισμός πιθανών outputs του Notebook 6
# --------------------------------------------

notebook6_processed_inventory = build_file_inventory(
    directory=PROCESSED_DIR,
    pattern="notebook6*",
    asset_type="processed_output",
)

notebook6_plots_inventory = build_file_inventory(
    directory=PLOTS_DIR,
    pattern="notebook6*",
    asset_type="plot_asset",
)

notebook6_assets_inventory = pd.concat(
    [notebook6_processed_inventory, notebook6_plots_inventory],
    axis=0,
    ignore_index=True,
)

# --------------------------------------------
# 2. Βασικός έλεγχος ότι βρέθηκαν αρχεία
# --------------------------------------------

if notebook6_assets_inventory.empty:
    raise FileNotFoundError(
        "Δεν εντοπίστηκαν αρχεία που να ξεκινούν από 'notebook6' "
        "στους φακέλους data/processed/ και plots/."
    )

# --------------------------------------------
# 3. Ταξινόμηση και εμφάνιση inventory
# --------------------------------------------

notebook6_assets_inventory = notebook6_assets_inventory.sort_values(
    by=["asset_type", "file_name"]
).reset_index(drop=True)

print("Εντοπίστηκαν τα παρακάτω assets του Notebook 6:")
display(notebook6_assets_inventory)

# --------------------------------------------
# 4. Ξεχωριστή γρήγορη σύνοψη ανά τύπο
# --------------------------------------------

inventory_summary = (
    notebook6_assets_inventory
    .groupby(["asset_type", "suffix"], dropna=False)
    .size()
    .reset_index(name="n_files")
    .sort_values(["asset_type", "suffix"])
    .reset_index(drop=True)
)

print("Σύνοψη inventory:")
display(inventory_summary)

Εντοπίστηκαν τα παρακάτω assets του Notebook 6:


,asset_type,file_name,suffix,parent_dir,absolute_path
0,plot_asset,notebook6_fuel_type_mix_stacked_bar.png,.png,plots,C:\Users\athin\Desktop\Car-Market-Analysis-Att...
1,plot_asset,notebook6_make_mix_stacked_bar.png,.png,plots,C:\Users\athin\Desktop\Car-Market-Analysis-Att...
2,plot_asset,notebook6_price_segment_share_bar.png,.png,plots,C:\Users\athin\Desktop\Car-Market-Analysis-Att...
3,plot_asset,notebook6_transmission_mix_stacked_bar.png,.png,plots,C:\Users\athin\Desktop\Car-Market-Analysis-Att...
4,processed_output,notebook6_fuel_type_dashboard_summary.csv,.csv,processed,C:\Users\athin\Desktop\Car-Market-Analysis-Att...
5,processed_output,notebook6_fuel_type_mix_plot_data.csv,.csv,processed,C:\Users\athin\Desktop\Car-Market-Analysis-Att...
6,processed_output,notebook6_make_dashboard_summary.csv,.csv,processed,C:\Users\athin\Desktop\Car-Market-Analysis-Att...
7,processed_output,notebook6_make_mix_plot_data.csv,.csv,processed,C:\Users\athin\Desktop\Car-Market-Analysis-Att...
8,processed_output,notebook6_market_overview_kpi_summary.csv,.csv,processed,C:\Users\athin\Desktop\Car-Market-Analysis-Att...
9,processed_output,notebook6_price_segment_share_plot_data.csv,.csv,processed,C:\Users\athin\Desktop\Car-Market-Analysis-Att...


Σύνοψη inventory:


,asset_type,suffix,n_files
0,plot_asset,.png,4
1,processed_output,.csv,11


In [4]:
# ============================================
# Notebook 7 - Στοχευμένη φόρτωση και validation των βασικών inputs
# ============================================

def resolve_single_file_by_pattern(directory: Path, pattern: str) -> Path:
    """
    Εντοπίζει ακριβώς ένα αρχείο με βάση glob pattern.

    Αν δεν βρεθεί κανένα ή αν βρεθούν περισσότερα από ένα,
    εμφανίζεται καθαρό σφάλμα για να αποφευχθεί ambiguity.
    """
    matched_files = sorted(directory.glob(pattern))

    if not matched_files:
        raise FileNotFoundError(
            f"Δεν βρέθηκε αρχείο με pattern '{pattern}' στον φάκελο: {directory}"
        )

    if len(matched_files) > 1:
        matched_names = [file_path.name for file_path in matched_files]
        raise ValueError(
            f"Βρέθηκαν περισσότερα από ένα αρχεία για pattern '{pattern}': {matched_names}"
        )

    return matched_files[0]


# --------------------------------------------
# 1. Βασικά CSV αρχεία που περιμένουμε από το Notebook 6
# --------------------------------------------

required_csv_files = {
    "market_overview_kpi_summary": PROCESSED_DIR / "notebook6_market_overview_kpi_summary.csv",
    "segment_kpi_summary": PROCESSED_DIR / "notebook6_segment_kpi_summary.csv",
    "fuel_type_dashboard_summary": PROCESSED_DIR / "notebook6_fuel_type_dashboard_summary.csv",
    "transmission_dashboard_summary": PROCESSED_DIR / "notebook6_transmission_dashboard_summary.csv",
    "make_dashboard_summary": PROCESSED_DIR / "notebook6_make_dashboard_summary.csv",
    "price_segment_share_plot_data": PROCESSED_DIR / "notebook6_price_segment_share_plot_data.csv",
    "fuel_type_mix_plot_data": PROCESSED_DIR / "notebook6_fuel_type_mix_plot_data.csv",
    "transmission_mix_plot_data": PROCESSED_DIR / "notebook6_transmission_mix_plot_data.csv",
    "make_mix_plot_data": PROCESSED_DIR / "notebook6_make_mix_plot_data.csv",
}

# Τα corrected make-by-segment tables τα εντοπίζουμε με pattern,
# ώστε να είμαστε ανθεκτικοί σε μικρές διαφορές στο filename.
required_csv_files["top10_make_by_segment_counts_corrected"] = resolve_single_file_by_pattern(
    PROCESSED_DIR,
    "notebook6_top10_make_by_segment_counts_correct*.csv",
)

required_csv_files["top10_make_by_segment_shares_corrected"] = resolve_single_file_by_pattern(
    PROCESSED_DIR,
    "notebook6_top10_make_by_segment_shares_correct*.csv",
)

# --------------------------------------------
# 2. Βασικά plot assets που περιμένουμε από το Notebook 6
# --------------------------------------------

required_plot_files = {
    "price_segment_share_bar": PLOTS_DIR / "notebook6_price_segment_share_bar.png",
    "fuel_type_mix_stacked_bar": PLOTS_DIR / "notebook6_fuel_type_mix_stacked_bar.png",
    "transmission_mix_stacked_bar": PLOTS_DIR / "notebook6_transmission_mix_stacked_bar.png",
    "make_mix_stacked_bar": PLOTS_DIR / "notebook6_make_mix_stacked_bar.png",
}

# --------------------------------------------
# 3. Validation ύπαρξης όλων των αρχείων
# --------------------------------------------

missing_csv_files = [
    file_path.name for file_path in required_csv_files.values()
    if not file_path.exists()
]

missing_plot_files = [
    file_path.name for file_path in required_plot_files.values()
    if not file_path.exists()
]

if missing_csv_files:
    raise FileNotFoundError(
        f"Λείπουν τα παρακάτω απαιτούμενα CSV αρχεία: {missing_csv_files}"
    )

if missing_plot_files:
    raise FileNotFoundError(
        f"Λείπουν τα παρακάτω απαιτούμενα plot assets: {missing_plot_files}"
    )

# --------------------------------------------
# 4. Φόρτωση των CSV αρχείων σε λεξικό DataFrames
# --------------------------------------------

loaded_tables = {
    table_key: read_csv_strict(file_path)
    for table_key, file_path in required_csv_files.items()
}

plot_asset_paths = required_plot_files.copy()

# --------------------------------------------
# 5. Δημιουργία σύντομου inventory των loaded inputs
# --------------------------------------------

loaded_input_records = []

for table_key, file_path in required_csv_files.items():
    df = loaded_tables[table_key]
    loaded_input_records.append(
        {
            "asset_key": table_key,
            "asset_kind": "csv_table",
            "file_name": file_path.name,
            "n_rows": df.shape[0],
            "n_columns": df.shape[1],
        }
    )

for plot_key, file_path in required_plot_files.items():
    loaded_input_records.append(
        {
            "asset_key": plot_key,
            "asset_kind": "plot_asset",
            "file_name": file_path.name,
            "n_rows": np.nan,
            "n_columns": np.nan,
        }
    )

loaded_inputs_inventory = (
    pd.DataFrame(loaded_input_records)
    .sort_values(["asset_kind", "asset_key"])
    .reset_index(drop=True)
)

print("Η στοχευμένη φόρτωση των βασικών inputs ολοκληρώθηκε επιτυχώς.")
display(loaded_inputs_inventory)

# --------------------------------------------
# 6. Δημιουργία ξεχωριστών DataFrames για ευκολότερη χρήση
# --------------------------------------------

market_overview_kpi_summary = loaded_tables["market_overview_kpi_summary"].copy()
segment_kpi_summary = loaded_tables["segment_kpi_summary"].copy()
fuel_type_dashboard_summary = loaded_tables["fuel_type_dashboard_summary"].copy()
transmission_dashboard_summary = loaded_tables["transmission_dashboard_summary"].copy()
make_dashboard_summary = loaded_tables["make_dashboard_summary"].copy()

price_segment_share_plot_data = loaded_tables["price_segment_share_plot_data"].copy()
fuel_type_mix_plot_data = loaded_tables["fuel_type_mix_plot_data"].copy()
transmission_mix_plot_data = loaded_tables["transmission_mix_plot_data"].copy()
make_mix_plot_data = loaded_tables["make_mix_plot_data"].copy()

top10_make_by_segment_counts_corrected = loaded_tables["top10_make_by_segment_counts_corrected"].copy()
top10_make_by_segment_shares_corrected = loaded_tables["top10_make_by_segment_shares_corrected"].copy()

print("\nShapes των βασικών tables:")
print("market_overview_kpi_summary:", market_overview_kpi_summary.shape)
print("segment_kpi_summary:", segment_kpi_summary.shape)
print("fuel_type_dashboard_summary:", fuel_type_dashboard_summary.shape)
print("transmission_dashboard_summary:", transmission_dashboard_summary.shape)
print("make_dashboard_summary:", make_dashboard_summary.shape)
print("price_segment_share_plot_data:", price_segment_share_plot_data.shape)
print("fuel_type_mix_plot_data:", fuel_type_mix_plot_data.shape)
print("transmission_mix_plot_data:", transmission_mix_plot_data.shape)
print("make_mix_plot_data:", make_mix_plot_data.shape)
print("top10_make_by_segment_counts_corrected:", top10_make_by_segment_counts_corrected.shape)
print("top10_make_by_segment_shares_corrected:", top10_make_by_segment_shares_corrected.shape)

Η στοχευμένη φόρτωση των βασικών inputs ολοκληρώθηκε επιτυχώς.


,asset_key,asset_kind,file_name,n_rows,n_columns
0,fuel_type_dashboard_summary,csv_table,notebook6_fuel_type_dashboard_summary.csv,36.00,8.00
1,fuel_type_mix_plot_data,csv_table,notebook6_fuel_type_mix_plot_data.csv,36.00,8.00
2,make_dashboard_summary,csv_table,notebook6_make_dashboard_summary.csv,40.00,8.00
3,make_mix_plot_data,csv_table,notebook6_make_mix_plot_data.csv,40.00,8.00
4,market_overview_kpi_summary,csv_table,notebook6_market_overview_kpi_summary.csv,11.00,8.00
5,price_segment_share_plot_data,csv_table,notebook6_price_segment_share_plot_data.csv,4.00,5.00
6,segment_kpi_summary,csv_table,notebook6_segment_kpi_summary.csv,48.00,9.00
7,top10_make_by_segment_counts_corrected,csv_table,notebook6_top10_make_by_segment_counts_correct...,4.00,11.00
8,top10_make_by_segment_shares_corrected,csv_table,notebook6_top10_make_by_segment_shares_correct...,4.00,11.00
9,transmission_dashboard_summary,csv_table,notebook6_transmission_dashboard_summary.csv,12.00,8.00



Shapes των βασικών tables:
market_overview_kpi_summary: (11, 8)
segment_kpi_summary: (48, 9)
fuel_type_dashboard_summary: (36, 8)
transmission_dashboard_summary: (12, 8)
make_dashboard_summary: (40, 8)
price_segment_share_plot_data: (4, 5)
fuel_type_mix_plot_data: (36, 8)
transmission_mix_plot_data: (12, 8)
make_mix_plot_data: (40, 8)
top10_make_by_segment_counts_corrected: (4, 11)
top10_make_by_segment_shares_corrected: (4, 11)


## Πρώτος έλεγχος συνέπειας των dashboard inputs

Μετά τη φόρτωση των βασικών αρχείων του Notebook 6, ακολουθεί ένας πρώτος
έλεγχος συνέπειας των εισόδων που θα χρησιμοποιηθούν στο παρόν notebook.

Στο στάδιο αυτό θα εξεταστούν:
- οι διαθέσιμες στήλες των βασικών dashboard tables,
- η ύπαρξη των απαιτούμενων πεδίων,
- και η γενική καταλληλότητα των loaded inputs για downstream χρήση
  σε sections, KPI cards, chart registries και streamlit-ready δομές.

In [5]:
# ============================================
# Notebook 7 - Πρώτος έλεγχος συνέπειας των loaded tables
# ============================================

# --------------------------------------------
# 1. Ορισμός των βασικών required columns
# --------------------------------------------

required_columns_by_table = {
    "market_overview_kpi_summary": [
        "kpi_id",
        "kpi_label_el",
        "group",
        "source_metric",
        "raw_value",
        "display_value",
        "value_type",
        "source_table",
    ],
    "segment_kpi_summary": [
        "price_segment",
        "kpi_id",
        "kpi_label_el",
        "group",
        "source_metric",
        "raw_value",
        "display_value",
        "value_type",
        "source_table",
    ],
    "fuel_type_dashboard_summary": [
        "price_segment",
        "fuel_type",
        "n_observations",
        "share_pct",
        "display_count",
        "display_share",
        "dimension",
        "source_table",
    ],
    "transmission_dashboard_summary": [
        "price_segment",
        "transmission_type",
        "n_observations",
        "share_pct",
        "display_count",
        "display_share",
        "dimension",
        "source_table",
    ],
    "make_dashboard_summary": [
        "price_segment",
        "make",
        "n_observations",
        "share_pct",
        "display_count",
        "display_share",
        "dimension",
        "source_table",
    ],
    "price_segment_share_plot_data": [
        "price_segment",
        "n_observations",
        "share_pct",
        "display_share",
        "display_count",
    ],
    "fuel_type_mix_plot_data": [
        "price_segment",
        "fuel_type",
        "n_observations",
        "share_pct",
        "display_count",
        "display_share",
        "dimension",
        "source_table",
    ],
    "transmission_mix_plot_data": [
        "price_segment",
        "transmission_type",
        "n_observations",
        "share_pct",
        "display_count",
        "display_share",
        "dimension",
        "source_table",
    ],
    "make_mix_plot_data": [
        "price_segment",
        "make",
        "n_observations",
        "share_pct",
        "display_count",
        "display_share",
        "dimension",
        "source_table",
    ],
    "top10_make_by_segment_counts_corrected": [
        "price_segment",
    ],
    "top10_make_by_segment_shares_corrected": [
        "price_segment",
    ],
}

# --------------------------------------------
# 2. Mapping των loaded tables για audit
# --------------------------------------------

table_registry = {
    "market_overview_kpi_summary": market_overview_kpi_summary,
    "segment_kpi_summary": segment_kpi_summary,
    "fuel_type_dashboard_summary": fuel_type_dashboard_summary,
    "transmission_dashboard_summary": transmission_dashboard_summary,
    "make_dashboard_summary": make_dashboard_summary,
    "price_segment_share_plot_data": price_segment_share_plot_data,
    "fuel_type_mix_plot_data": fuel_type_mix_plot_data,
    "transmission_mix_plot_data": transmission_mix_plot_data,
    "make_mix_plot_data": make_mix_plot_data,
    "top10_make_by_segment_counts_corrected": top10_make_by_segment_counts_corrected,
    "top10_make_by_segment_shares_corrected": top10_make_by_segment_shares_corrected,
}

# --------------------------------------------
# 3. Validation και schema audit
# --------------------------------------------

schema_audit_records = []

for table_name, df in table_registry.items():
    required_columns = required_columns_by_table.get(table_name, [])
    existing_columns = df.columns.tolist()
    missing_columns = [col for col in required_columns if col not in existing_columns]

    schema_audit_records.append(
        {
            "table_name": table_name,
            "n_rows": df.shape[0],
            "n_columns": df.shape[1],
            "has_missing_required_columns": len(missing_columns) > 0,
            "missing_required_columns": ", ".join(missing_columns) if missing_columns else "",
            "columns": " | ".join(existing_columns),
        }
    )

schema_audit_df = pd.DataFrame(schema_audit_records).sort_values("table_name").reset_index(drop=True)

# Αν κάποιο table δεν έχει required columns, σταματάμε εδώ.
tables_with_missing_columns = schema_audit_df[schema_audit_df["has_missing_required_columns"]]

if not tables_with_missing_columns.empty:
    display(schema_audit_df)
    raise ValueError(
        "Εντοπίστηκαν tables με ελλείπουσες required columns. "
        "Έλεγξε το schema audit πριν συνεχίσουμε."
    )

print("Ο πρώτος έλεγχος συνέπειας των στηλών ολοκληρώθηκε επιτυχώς.")
display(schema_audit_df[["table_name", "n_rows", "n_columns", "has_missing_required_columns", "missing_required_columns"]])

# --------------------------------------------
# 4. Προαιρετικό preview των πρώτων γραμμών
# --------------------------------------------

print("\nPreview - market_overview_kpi_summary")
display(market_overview_kpi_summary.head())

print("\nPreview - segment_kpi_summary")
display(segment_kpi_summary.head())

print("\nPreview - fuel_type_dashboard_summary")
display(fuel_type_dashboard_summary.head())

Ο πρώτος έλεγχος συνέπειας των στηλών ολοκληρώθηκε επιτυχώς.


,table_name,n_rows,n_columns,has_missing_required_columns,missing_required_columns
0,fuel_type_dashboard_summary,36,8,False,
1,fuel_type_mix_plot_data,36,8,False,
2,make_dashboard_summary,40,8,False,
3,make_mix_plot_data,40,8,False,
4,market_overview_kpi_summary,11,8,False,
5,price_segment_share_plot_data,4,5,False,
6,segment_kpi_summary,48,9,False,
7,top10_make_by_segment_counts_corrected,4,11,False,
8,top10_make_by_segment_shares_corrected,4,11,False,
9,transmission_dashboard_summary,12,8,False,



Preview - market_overview_kpi_summary


,kpi_id,kpi_label_el,group,source_metric,raw_value,display_value,value_type,source_table
0,market_n_observations,Συνολικός αριθμός παρατηρήσεων,Μέγεθος αγοράς,n_observations,"8,296.00","8,296",integer,notebook5_market_overview
1,market_n_variables,Συνολικός αριθμός μεταβλητών,Δομή δεδομένων,n_variables,12.00,12,integer,notebook5_market_overview
2,market_n_unique_makes,Διακριτοί κατασκευαστές,Ποικιλία αγοράς,n_unique_makes,71.00,71,integer,notebook5_market_overview
3,market_n_unique_models,Διακριτά μοντέλα,Ποικιλία αγοράς,n_unique_models,554.00,554,integer,notebook5_market_overview
4,market_n_unique_regions,Διακριτές περιοχές,Γεωγραφική κάλυψη,n_unique_regions,276.00,276,integer,notebook5_market_overview



Preview - segment_kpi_summary


,price_segment,kpi_id,kpi_label_el,group,source_metric,raw_value,display_value,value_type,source_table
0,Χαμηλή,segment_Χαμηλή_n_observations,Αριθμός παρατηρήσεων,Μέγεθος segment,n_observations,"2,077.00","2,077",integer,notebook5_price_segment_counts + notebook5_seg...
1,Χαμηλή,segment_Χαμηλή_share_pct,Ποσοστό συμμετοχής,Μέγεθος segment,share_pct,25.04,25.04%,percentage,notebook5_price_segment_counts + notebook5_seg...
2,Χαμηλή,segment_Χαμηλή_median_horsepower,Διάμεση ιπποδύναμη,Τεχνικά χαρακτηριστικά,median_horsepower,100.00,100 hp,horsepower,notebook5_price_segment_counts + notebook5_seg...
3,Χαμηλή,segment_Χαμηλή_median_engine_size,Διάμεσος κυβισμός,Τεχνικά χαρακτηριστικά,median_engine_size,"1,200.00","1,200 cc",engine_size,notebook5_price_segment_counts + notebook5_seg...
4,Χαμηλή,segment_Χαμηλή_mean_horsepower,Μέση ιπποδύναμη,Τεχνικά χαρακτηριστικά,mean_horsepower,94.60,95 hp,horsepower,notebook5_price_segment_counts + notebook5_seg...



Preview - fuel_type_dashboard_summary


,price_segment,fuel_type,n_observations,share_pct,display_count,display_share,dimension,source_table
0,Χαμηλή,Βενζίνη,1084,52.19,"1,084",52.19%,fuel_type,notebook5_fuel_type_by_segment_counts + notebo...
1,Χαμηλή,Πετρέλαιο,568,27.35,568,27.35%,fuel_type,notebook5_fuel_type_by_segment_counts + notebo...
2,Χαμηλή,Υβριδικό Βενζίνης,275,13.24,275,13.24%,fuel_type,notebook5_fuel_type_by_segment_counts + notebo...
3,Χαμηλή,Plug-in Hybrid Βενζίνης,6,0.29,6,0.29%,fuel_type,notebook5_fuel_type_by_segment_counts + notebo...
4,Χαμηλή,Ηλεκτρικό,90,4.33,90,4.33%,fuel_type,notebook5_fuel_type_by_segment_counts + notebo...


## Τυποποίηση του `price_segment` και σταθερή σειρά κατηγοριών

Για να διασφαλιστεί η συνέπεια μεταξύ όλων των dashboard inputs, στο παρόν στάδιο
τυποποιείται η μεταβλητή `price_segment` στα tables όπου αυτή υπάρχει.

Ο στόχος είναι:
- να αποφεύγονται ασυνέπειες σε labels ή κρυφούς χαρακτήρες,
- να επιβληθεί μία ενιαία και λογική σειρά κατηγοριών,
- και να προετοιμαστούν τα δεδομένα για σωστή ταξινόμηση σε cards, tables και charts.

Η canonical σειρά των segments στο παρόν project είναι:
1. Χαμηλή
2. Χαμηλομεσαία
3. Μεσοϋψηλή
4. Υψηλή

In [6]:
# ============================================
# Notebook 7 - Τυποποίηση του price_segment
# ============================================

# --------------------------------------------
# 1. Canonical σειρά price segments
# --------------------------------------------

canonical_segment_order = [
    "Χαμηλή",
    "Χαμηλομεσαία",
    "Μεσοϋψηλή",
    "Υψηλή",
]

segment_label_mapping = {
    "Χαμηλή": "Χαμηλή",
    "Χαμηλομεσαία": "Χαμηλομεσαία",
    "Μεσουψηλή": "Μεσοϋψηλή",
    "Μεσοϋψηλή": "Μεσοϋψηλή",
    "Υψηλή": "Υψηλή",
}

# --------------------------------------------
# 2. Helper function για τυποποίηση labels
# --------------------------------------------

def standardize_price_segment_labels(
    df: pd.DataFrame,
    column: str = "price_segment",
) -> pd.DataFrame:
    """
    Τυποποιεί τις τιμές του price_segment και επιβάλλει
    ordered categorical dtype με canonical σειρά.
    """
    cleaned_df = df.copy()

    if column not in cleaned_df.columns:
        return cleaned_df

    cleaned_series = (
        cleaned_df[column]
        .astype("string")
        .str.replace("\u00a0", " ", regex=False)
        .str.replace("\u200b", "", regex=False)
        .str.strip()
        .replace(segment_label_mapping)
    )

    cleaned_series = cleaned_series.replace({
        "nan": pd.NA,
        "None": pd.NA,
        "<NA>": pd.NA,
    })

    cleaned_df[column] = pd.Categorical(
        cleaned_series,
        categories=canonical_segment_order,
        ordered=True,
    )

    return cleaned_df


# --------------------------------------------
# 3. Tables που περιέχουν price_segment
# --------------------------------------------

tables_with_price_segment = [
    "segment_kpi_summary",
    "fuel_type_dashboard_summary",
    "transmission_dashboard_summary",
    "make_dashboard_summary",
    "price_segment_share_plot_data",
    "fuel_type_mix_plot_data",
    "transmission_mix_plot_data",
    "make_mix_plot_data",
    "top10_make_by_segment_counts_corrected",
    "top10_make_by_segment_shares_corrected",
]

# --------------------------------------------
# 4. Εφαρμογή της τυποποίησης
# --------------------------------------------

for table_name in tables_with_price_segment:
    loaded_tables[table_name] = standardize_price_segment_labels(
        loaded_tables[table_name],
        column="price_segment",
    )

# Επανεκχώρηση για να μείνουν συγχρονισμένα τα ξεχωριστά DataFrames
segment_kpi_summary = loaded_tables["segment_kpi_summary"].copy()
fuel_type_dashboard_summary = loaded_tables["fuel_type_dashboard_summary"].copy()
transmission_dashboard_summary = loaded_tables["transmission_dashboard_summary"].copy()
make_dashboard_summary = loaded_tables["make_dashboard_summary"].copy()
price_segment_share_plot_data = loaded_tables["price_segment_share_plot_data"].copy()
fuel_type_mix_plot_data = loaded_tables["fuel_type_mix_plot_data"].copy()
transmission_mix_plot_data = loaded_tables["transmission_mix_plot_data"].copy()
make_mix_plot_data = loaded_tables["make_mix_plot_data"].copy()
top10_make_by_segment_counts_corrected = loaded_tables["top10_make_by_segment_counts_corrected"].copy()
top10_make_by_segment_shares_corrected = loaded_tables["top10_make_by_segment_shares_corrected"].copy()

# --------------------------------------------
# 5. Έλεγχος των τελικών labels ανά table
# --------------------------------------------

segment_validation_records = []

for table_name in tables_with_price_segment:
    df = loaded_tables[table_name]

    unique_segments = [
        str(value)
        for value in df["price_segment"].dropna().unique().tolist()
    ]

    has_missing_segment = df["price_segment"].isna().any()

    segment_validation_records.append(
        {
            "table_name": table_name,
            "n_unique_segments": len(unique_segments),
            "segments_found": " | ".join(unique_segments),
            "has_missing_segment": has_missing_segment,
        }
    )

segment_validation_df = pd.DataFrame(segment_validation_records).sort_values("table_name").reset_index(drop=True)

print("Η τυποποίηση του price_segment ολοκληρώθηκε.")
display(segment_validation_df)

Η τυποποίηση του price_segment ολοκληρώθηκε.


,table_name,n_unique_segments,segments_found,has_missing_segment
0,fuel_type_dashboard_summary,4,Χαμηλή | Χαμηλομεσαία | Μεσοϋψηλή | Υψηλή,False
1,fuel_type_mix_plot_data,4,Χαμηλή | Χαμηλομεσαία | Μεσοϋψηλή | Υψηλή,False
2,make_dashboard_summary,4,Χαμηλή | Χαμηλομεσαία | Μεσοϋψηλή | Υψηλή,False
3,make_mix_plot_data,4,Χαμηλή | Χαμηλομεσαία | Μεσοϋψηλή | Υψηλή,False
4,price_segment_share_plot_data,4,Χαμηλή | Χαμηλομεσαία | Μεσοϋψηλή | Υψηλή,False
5,segment_kpi_summary,4,Χαμηλή | Χαμηλομεσαία | Μεσοϋψηλή | Υψηλή,False
6,top10_make_by_segment_counts_corrected,4,Χαμηλή | Χαμηλομεσαία | Μεσοϋψηλή | Υψηλή,False
7,top10_make_by_segment_shares_corrected,4,Χαμηλή | Χαμηλομεσαία | Μεσοϋψηλή | Υψηλή,False
8,transmission_dashboard_summary,4,Χαμηλή | Χαμηλομεσαία | Μεσοϋψηλή | Υψηλή,False
9,transmission_mix_plot_data,4,Χαμηλή | Χαμηλομεσαία | Μεσοϋψηλή | Υψηλή,False


## Σταθερή ταξινόμηση των dashboard tables

Αφού τυποποιήθηκε το `price_segment`, ακολουθεί η επιβολή σταθερής ταξινόμησης
στα βασικά dashboard tables.

Η διαδικασία αυτή είναι σημαντική ώστε:
- τα summaries να εμφανίζονται με συνεπή σειρά,
- τα KPI blocks και τα registries να βασίζονται σε προβλέψιμη δομή,
- και τα μελλοντικά dashboard components να μην εξαρτώνται από τυχαία σειρά γραμμών.

In [7]:
# ============================================
# Notebook 7 - Σταθερή ταξινόμηση των dashboard tables
# ============================================

def sort_dashboard_table(table_name: str, df: pd.DataFrame) -> pd.DataFrame:
    """
    Εφαρμόζει σταθερή ταξινόμηση ανάλογα με το είδος του dashboard table.
    """
    sort_rules = {
        "segment_kpi_summary": ["price_segment", "group", "kpi_label_el"],
        "fuel_type_dashboard_summary": ["price_segment", "fuel_type"],
        "transmission_dashboard_summary": ["price_segment", "transmission_type"],
        "make_dashboard_summary": ["price_segment", "make"],
        "price_segment_share_plot_data": ["price_segment"],
        "fuel_type_mix_plot_data": ["price_segment", "fuel_type"],
        "transmission_mix_plot_data": ["price_segment", "transmission_type"],
        "make_mix_plot_data": ["price_segment", "make"],
        "top10_make_by_segment_counts_corrected": ["price_segment"],
        "top10_make_by_segment_shares_corrected": ["price_segment"],
    }

    sort_columns = [
        column_name
        for column_name in sort_rules.get(table_name, [])
        if column_name in df.columns
    ]

    if not sort_columns:
        return df.reset_index(drop=True)

    return df.sort_values(sort_columns).reset_index(drop=True)


# --------------------------------------------
# 1. Tables που θέλουμε να ταξινομήσουμε
# --------------------------------------------

tables_to_sort = [
    "segment_kpi_summary",
    "fuel_type_dashboard_summary",
    "transmission_dashboard_summary",
    "make_dashboard_summary",
    "price_segment_share_plot_data",
    "fuel_type_mix_plot_data",
    "transmission_mix_plot_data",
    "make_mix_plot_data",
    "top10_make_by_segment_counts_corrected",
    "top10_make_by_segment_shares_corrected",
]

# --------------------------------------------
# 2. Εφαρμογή της ταξινόμησης
# --------------------------------------------

for table_name in tables_to_sort:
    loaded_tables[table_name] = sort_dashboard_table(
        table_name=table_name,
        df=loaded_tables[table_name],
    )

# Επανεκχώρηση των DataFrames μετά την ταξινόμηση
segment_kpi_summary = loaded_tables["segment_kpi_summary"].copy()
fuel_type_dashboard_summary = loaded_tables["fuel_type_dashboard_summary"].copy()
transmission_dashboard_summary = loaded_tables["transmission_dashboard_summary"].copy()
make_dashboard_summary = loaded_tables["make_dashboard_summary"].copy()
price_segment_share_plot_data = loaded_tables["price_segment_share_plot_data"].copy()
fuel_type_mix_plot_data = loaded_tables["fuel_type_mix_plot_data"].copy()
transmission_mix_plot_data = loaded_tables["transmission_mix_plot_data"].copy()
make_mix_plot_data = loaded_tables["make_mix_plot_data"].copy()
top10_make_by_segment_counts_corrected = loaded_tables["top10_make_by_segment_counts_corrected"].copy()
top10_make_by_segment_shares_corrected = loaded_tables["top10_make_by_segment_shares_corrected"].copy()

# --------------------------------------------
# 3. Γρήγορο preview μετά την ταξινόμηση
# --------------------------------------------

print("Η σταθερή ταξινόμηση των dashboard tables ολοκληρώθηκε επιτυχώς.\n")

print("Preview - price_segment_share_plot_data")
display(price_segment_share_plot_data.head())

print("\nPreview - fuel_type_dashboard_summary")
display(fuel_type_dashboard_summary.head())

print("\nPreview - transmission_dashboard_summary")
display(transmission_dashboard_summary.head())

print("\nPreview - make_dashboard_summary")
display(make_dashboard_summary.head())

Η σταθερή ταξινόμηση των dashboard tables ολοκληρώθηκε επιτυχώς.

Preview - price_segment_share_plot_data


,price_segment,n_observations,share_pct,display_share,display_count
0,Χαμηλή,2077,25.04,25.04%,"2,077"
1,Χαμηλομεσαία,2081,25.08,25.08%,"2,081"
2,Μεσοϋψηλή,2069,24.94,24.94%,"2,069"
3,Υψηλή,2069,24.94,24.94%,"2,069"



Preview - fuel_type_dashboard_summary


,price_segment,fuel_type,n_observations,share_pct,display_count,display_share,dimension,source_table
0,Χαμηλή,CNG / Βενζίνη,19,0.91,19,0.91%,fuel_type,notebook5_fuel_type_by_segment_counts + notebo...
1,Χαμηλή,LPG / Βενζίνη,32,1.54,32,1.54%,fuel_type,notebook5_fuel_type_by_segment_counts + notebo...
2,Χαμηλή,Plug-in Hybrid Βενζίνης,6,0.29,6,0.29%,fuel_type,notebook5_fuel_type_by_segment_counts + notebo...
3,Χαμηλή,Plug-in Hybrid Πετρελαίου,0,0.00,0,0.00%,fuel_type,notebook5_fuel_type_by_segment_counts + notebo...
4,Χαμηλή,Βενζίνη,1084,52.19,"1,084",52.19%,fuel_type,notebook5_fuel_type_by_segment_counts + notebo...



Preview - transmission_dashboard_summary


,price_segment,transmission_type,n_observations,share_pct,display_count,display_share,dimension,source_table
0,Χαμηλή,Αυτόματο,371,17.86,371,17.86%,transmission,notebook5_transmission_by_segment_counts + not...
1,Χαμηλή,Ημιαυτόματο,1,0.05,1,0.05%,transmission,notebook5_transmission_by_segment_counts + not...
2,Χαμηλή,Χειροκίνητο,1705,82.09,"1,705",82.09%,transmission,notebook5_transmission_by_segment_counts + not...
3,Χαμηλομεσαία,Αυτόματο,1058,50.84,"1,058",50.84%,transmission,notebook5_transmission_by_segment_counts + not...
4,Χαμηλομεσαία,Ημιαυτόματο,1,0.05,1,0.05%,transmission,notebook5_transmission_by_segment_counts + not...



Preview - make_dashboard_summary


,price_segment,make,n_observations,share_pct,display_count,display_share,dimension,source_table
0,Χαμηλή,Audi,3,0.23,3,0.23%,make,notebook6_top10_make_by_segment_counts_correct...
1,Χαμηλή,BMW,0,0.00,0,0.00%,make,notebook6_top10_make_by_segment_counts_correct...
2,Χαμηλή,Citroen,293,22.52,293,22.52%,make,notebook6_top10_make_by_segment_counts_correct...
3,Χαμηλή,Ford,74,5.69,74,5.69%,make,notebook6_top10_make_by_segment_counts_correct...
4,Χαμηλή,Hyundai,129,9.92,129,9.92%,make,notebook6_top10_make_by_segment_counts_correct...


## Ορισμός των dashboard sections / views

Στο παρόν στάδιο ορίζονται οι βασικές ενότητες του μελλοντικού dashboard,
ώστε το project να αποκτήσει μία σαφή και τεκμηριωμένη presentation structure.

Οι ενότητες αυτές δεν αποτελούν απλώς οπτικές κατηγορίες, αλλά λειτουργούν ως
δομικά στοιχεία για:
- τη χαρτογράφηση των διαθέσιμων tables και charts,
- την οργάνωση KPI cards και summaries,
- και τη μελλοντική υλοποίηση multipage ή multi-section Streamlit εφαρμογής.

Η λογική του dashboard οργανώνεται στις εξής κύριες ενότητες:
1. `market_overview`
2. `segment_profile`
3. `fuel_mix`
4. `transmission_mix`
5. `make_concentration`

In [8]:
# ============================================
# Notebook 7 - Dashboard sections / views registry
# ============================================

dashboard_sections_registry = pd.DataFrame(
    [
        {
            "section_order": 1,
            "section_id": "market_overview",
            "section_title_el": "Επισκόπηση αγοράς",
            "section_title_en": "Market Overview",
            "section_description_el": (
                "Συνολική εικόνα της αγοράς μεταχειρισμένων αυτοκινήτων "
                "μέσω βασικών δεικτών και market-level KPIs."
            ),
            "primary_table_key": "market_overview_kpi_summary",
            "primary_plot_key": pd.NA,
            "default_component_type": "kpi_cards",
            "supports_price_segment_filter": False,
            "supports_category_filter": False,
            "streamlit_tab_name": "market_overview",
        },
        {
            "section_order": 2,
            "section_id": "segment_profile",
            "section_title_el": "Προφίλ price segments",
            "section_title_en": "Segment Profile",
            "section_description_el": (
                "Συγκριτική παρουσίαση των βασικών KPIs ανά price segment, "
                "με έμφαση στο μέγεθος και στα τεχνικά χαρακτηριστικά."
            ),
            "primary_table_key": "segment_kpi_summary",
            "primary_plot_key": "price_segment_share_bar",
            "default_component_type": "kpi_cards_plus_chart",
            "supports_price_segment_filter": True,
            "supports_category_filter": False,
            "streamlit_tab_name": "segment_profile",
        },
        {
            "section_order": 3,
            "section_id": "fuel_mix",
            "section_title_el": "Σύνθεση καυσίμου",
            "section_title_en": "Fuel Mix",
            "section_description_el": (
                "Κατανομή και σύνθεση του τύπου καυσίμου ανά price segment."
            ),
            "primary_table_key": "fuel_type_dashboard_summary",
            "primary_plot_key": "fuel_type_mix_stacked_bar",
            "default_component_type": "stacked_bar_chart",
            "supports_price_segment_filter": True,
            "supports_category_filter": True,
            "streamlit_tab_name": "fuel_mix",
        },
        {
            "section_order": 4,
            "section_id": "transmission_mix",
            "section_title_el": "Σύνθεση μετάδοσης",
            "section_title_en": "Transmission Mix",
            "section_description_el": (
                "Κατανομή και σύνθεση του τύπου μετάδοσης ανά price segment."
            ),
            "primary_table_key": "transmission_dashboard_summary",
            "primary_plot_key": "transmission_mix_stacked_bar",
            "default_component_type": "stacked_bar_chart",
            "supports_price_segment_filter": True,
            "supports_category_filter": True,
            "streamlit_tab_name": "transmission_mix",
        },
        {
            "section_order": 5,
            "section_id": "make_concentration",
            "section_title_el": "Συγκέντρωση κατασκευαστών",
            "section_title_en": "Make Concentration",
            "section_description_el": (
                "Σύνθεση των βασικών κατασκευαστών ανά price segment "
                "και αποτύπωση της σχετικής συγκέντρωσης της αγοράς."
            ),
            "primary_table_key": "make_dashboard_summary",
            "primary_plot_key": "make_mix_stacked_bar",
            "default_component_type": "stacked_bar_chart",
            "supports_price_segment_filter": True,
            "supports_category_filter": True,
            "streamlit_tab_name": "make_concentration",
        },
    ]
).sort_values("section_order").reset_index(drop=True)

print("Το dashboard sections registry δημιουργήθηκε επιτυχώς.")
display(dashboard_sections_registry)

Το dashboard sections registry δημιουργήθηκε επιτυχώς.


,section_order,section_id,section_title_el,section_title_en,section_description_el,primary_table_key,primary_plot_key,default_component_type,supports_price_segment_filter,supports_category_filter,streamlit_tab_name
0,1,market_overview,Επισκόπηση αγοράς,Market Overview,Συνολική εικόνα της αγοράς μεταχειρισμένων αυτ...,market_overview_kpi_summary,NaN,kpi_cards,False,False,market_overview
1,2,segment_profile,Προφίλ price segments,Segment Profile,Συγκριτική παρουσίαση των βασικών KPIs ανά pri...,segment_kpi_summary,price_segment_share_bar,kpi_cards_plus_chart,True,False,segment_profile
2,3,fuel_mix,Σύνθεση καυσίμου,Fuel Mix,Κατανομή και σύνθεση του τύπου καυσίμου ανά pr...,fuel_type_dashboard_summary,fuel_type_mix_stacked_bar,stacked_bar_chart,True,True,fuel_mix
3,4,transmission_mix,Σύνθεση μετάδοσης,Transmission Mix,Κατανομή και σύνθεση του τύπου μετάδοσης ανά p...,transmission_dashboard_summary,transmission_mix_stacked_bar,stacked_bar_chart,True,True,transmission_mix
4,5,make_concentration,Συγκέντρωση κατασκευαστών,Make Concentration,Σύνθεση των βασικών κατασκευαστών ανά price se...,make_dashboard_summary,make_mix_stacked_bar,stacked_bar_chart,True,True,make_concentration


## Ορισμός cards / KPI registry

Στο παρόν στάδιο δημιουργείται ένα ενιαίο registry για τα KPI cards του dashboard.

Η λογική είναι διπλή:
- να οργανωθούν τα **market-level cards** της συνολικής αγοράς,
- και να οργανωθούν τα **segment-level cards** για κάθε `price_segment`.

Με τον τρόπο αυτό, τα KPI στοιχεία μετατρέπονται από απλά summary tables
σε δομές κατάλληλες για downstream dashboard rendering, με σταθερά ids,
σαφές scope χρήσης και προβλέψιμη σειρά εμφάνισης.

In [9]:
# ============================================
# Notebook 7 - Dashboard cards / KPI registry
# ============================================

# --------------------------------------------
# 1. Market-level cards
# --------------------------------------------

market_cards_registry = market_overview_kpi_summary.copy()

market_cards_registry["section_id"] = "market_overview"
market_cards_registry["card_scope"] = "market"
market_cards_registry["price_segment"] = "ALL"
market_cards_registry["component_type"] = "kpi_card"
market_cards_registry["card_order"] = np.arange(1, len(market_cards_registry) + 1)

market_cards_registry["card_id"] = (
    "card_market_" + market_cards_registry["kpi_id"].astype("string")
)

market_cards_registry = market_cards_registry[
    [
        "card_id",
        "section_id",
        "card_scope",
        "price_segment",
        "component_type",
        "card_order",
        "kpi_id",
        "kpi_label_el",
        "group",
        "source_metric",
        "raw_value",
        "display_value",
        "value_type",
        "source_table",
    ]
].copy()

# --------------------------------------------
# 2. Segment-level cards
# --------------------------------------------

segment_cards_registry = segment_kpi_summary.copy()

segment_cards_registry["section_id"] = "segment_profile"
segment_cards_registry["card_scope"] = "segment"
segment_cards_registry["component_type"] = "kpi_card"

segment_cards_registry["price_segment_str"] = (
    segment_cards_registry["price_segment"]
    .astype("string")
    .str.strip()
)

segment_cards_registry["card_order"] = (
    segment_cards_registry
    .groupby("price_segment_str")
    .cumcount()
    + 1
)

segment_cards_registry["card_id"] = (
    "card_segment_"
    + segment_cards_registry["price_segment_str"]
        .str.lower()
        .str.replace(" ", "_", regex=False)
        .str.replace("ά", "a", regex=False)
        .str.replace("ή", "i", regex=False)
        .str.replace("ί", "i", regex=False)
        .str.replace("ό", "o", regex=False)
        .str.replace("ύ", "y", regex=False)
        .str.replace("ώ", "o", regex=False)
        .str.replace("ϊ", "i", regex=False)
        .str.replace("ΐ", "i", regex=False)
        .str.replace("ϋ", "y", regex=False)
        .str.replace("ΰ", "y", regex=False)
    + "_"
    + segment_cards_registry["kpi_id"].astype("string")
)

segment_cards_registry = segment_cards_registry[
    [
        "card_id",
        "section_id",
        "card_scope",
        "price_segment",
        "component_type",
        "card_order",
        "kpi_id",
        "kpi_label_el",
        "group",
        "source_metric",
        "raw_value",
        "display_value",
        "value_type",
        "source_table",
    ]
].copy()

# --------------------------------------------
# 3. Ενοποίηση σε ένα registry
# --------------------------------------------

dashboard_cards_registry = pd.concat(
    [market_cards_registry, segment_cards_registry],
    axis=0,
    ignore_index=True,
)

dashboard_cards_registry = dashboard_cards_registry.sort_values(
    by=["section_id", "price_segment", "card_order", "kpi_label_el"],
    na_position="last",
).reset_index(drop=True)

# --------------------------------------------
# 4. Γρήγορος έλεγχος αποτελέσματος
# --------------------------------------------

cards_summary_df = (
    dashboard_cards_registry
    .groupby(["section_id", "card_scope"], dropna=False)
    .size()
    .reset_index(name="n_cards")
    .sort_values(["section_id", "card_scope"])
    .reset_index(drop=True)
)

print("Το dashboard cards registry δημιουργήθηκε επιτυχώς.\n")

print("Σύνοψη cards ανά section:")
display(cards_summary_df)

print("\nPreview - dashboard_cards_registry")
display(dashboard_cards_registry.head(15))

Το dashboard cards registry δημιουργήθηκε επιτυχώς.

Σύνοψη cards ανά section:


,section_id,card_scope,n_cards
0,market_overview,market,11
1,segment_profile,segment,48



Preview - dashboard_cards_registry


,card_id,section_id,card_scope,price_segment,component_type,card_order,kpi_id,kpi_label_el,group,source_metric,raw_value,display_value,value_type,source_table
0,card_market_market_n_observations,market_overview,market,ALL,kpi_card,1,market_n_observations,Συνολικός αριθμός παρατηρήσεων,Μέγεθος αγοράς,n_observations,"8,296.00","8,296",integer,notebook5_market_overview
1,card_market_market_n_variables,market_overview,market,ALL,kpi_card,2,market_n_variables,Συνολικός αριθμός μεταβλητών,Δομή δεδομένων,n_variables,12.00,12,integer,notebook5_market_overview
2,card_market_market_n_unique_makes,market_overview,market,ALL,kpi_card,3,market_n_unique_makes,Διακριτοί κατασκευαστές,Ποικιλία αγοράς,n_unique_makes,71.00,71,integer,notebook5_market_overview
3,card_market_market_n_unique_models,market_overview,market,ALL,kpi_card,4,market_n_unique_models,Διακριτά μοντέλα,Ποικιλία αγοράς,n_unique_models,554.00,554,integer,notebook5_market_overview
4,card_market_market_n_unique_regions,market_overview,market,ALL,kpi_card,5,market_n_unique_regions,Διακριτές περιοχές,Γεωγραφική κάλυψη,n_unique_regions,276.00,276,integer,notebook5_market_overview
5,card_market_market_median_price,market_overview,market,ALL,kpi_card,6,market_median_price,Διάμεση τιμή,Τιμή,median_price,"22,000.00","€22,000",currency,notebook5_market_overview
6,card_market_market_mean_price,market_overview,market,ALL,kpi_card,7,market_mean_price,Μέση τιμή,Τιμή,mean_price,"33,280.20","€33,280",currency,notebook5_market_overview
7,card_market_market_median_mileage,market_overview,market,ALL,kpi_card,8,market_median_mileage,Διάμεση χιλιομετρική κάλυψη,Χρήση οχήματος,median_mileage,"50,019.50","50,020 km",mileage,notebook5_market_overview
8,card_market_market_median_age,market_overview,market,ALL,kpi_card,9,market_median_age,Διάμεση ηλικία,Χρήση οχήματος,median_age,4.00,4.0 έτη,age,notebook5_market_overview
9,card_market_market_median_horsepower,market_overview,market,ALL,kpi_card,10,market_median_horsepower,Διάμεση ιπποδύναμη,Τεχνικά χαρακτηριστικά,median_horsepower,130.00,130 hp,horsepower,notebook5_market_overview


## Ορισμός chart registry

Στο παρόν στάδιο δημιουργείται ένα ενιαίο registry για τα βασικά charts του dashboard.

Ο στόχος είναι να οριστούν με σαφήνεια:
- το `chart_id` κάθε γραφήματος,
- η ενότητα (`section_id`) στην οποία ανήκει,
- το input table που το τροφοδοτεί,
- το σχετικό plot asset που έχει ήδη παραχθεί,
- και τα βασικά πεδία που θα χρειαστούν αργότερα για rendering ή επαναχρησιμοποίηση.

Με αυτόν τον τρόπο, τα διαθέσιμα charts παύουν να αποτελούν μεμονωμένα αρχεία
και ενσωματώνονται σε μία τεκμηριωμένη dashboard structure.

In [10]:
# ============================================
# Notebook 7 - Dashboard chart registry
# ============================================

dashboard_charts_registry = pd.DataFrame(
    [
        {
            "chart_order": 1,
            "chart_id": "chart_price_segment_share",
            "section_id": "segment_profile",
            "chart_title_el": "Κατανομή της αγοράς ανά price segment",
            "chart_title_en": "Market Share by Price Segment",
            "chart_type": "horizontal_bar",
            "data_table_key": "price_segment_share_plot_data",
            "plot_asset_key": "price_segment_share_bar",
            "x_field": "share_pct",
            "y_field": "price_segment",
            "category_field": pd.NA,
            "value_field": "n_observations",
            "supports_price_segment_filter": False,
            "supports_category_filter": False,
        },
        {
            "chart_order": 2,
            "chart_id": "chart_fuel_mix",
            "section_id": "fuel_mix",
            "chart_title_el": "Σύνθεση τύπου καυσίμου ανά price segment",
            "chart_title_en": "Fuel Type Mix by Price Segment",
            "chart_type": "stacked_horizontal_bar",
            "data_table_key": "fuel_type_mix_plot_data",
            "plot_asset_key": "fuel_type_mix_stacked_bar",
            "x_field": "share_pct",
            "y_field": "price_segment",
            "category_field": "fuel_type",
            "value_field": "n_observations",
            "supports_price_segment_filter": True,
            "supports_category_filter": True,
        },
        {
            "chart_order": 3,
            "chart_id": "chart_transmission_mix",
            "section_id": "transmission_mix",
            "chart_title_el": "Σύνθεση τύπου μετάδοσης ανά price segment",
            "chart_title_en": "Transmission Mix by Price Segment",
            "chart_type": "stacked_horizontal_bar",
            "data_table_key": "transmission_mix_plot_data",
            "plot_asset_key": "transmission_mix_stacked_bar",
            "x_field": "share_pct",
            "y_field": "price_segment",
            "category_field": "transmission_type",
            "value_field": "n_observations",
            "supports_price_segment_filter": True,
            "supports_category_filter": True,
        },
        {
            "chart_order": 4,
            "chart_id": "chart_make_mix",
            "section_id": "make_concentration",
            "chart_title_el": "Σύνθεση βασικών κατασκευαστών ανά price segment",
            "chart_title_en": "Make Mix by Price Segment",
            "chart_type": "stacked_horizontal_bar",
            "data_table_key": "make_mix_plot_data",
            "plot_asset_key": "make_mix_stacked_bar",
            "x_field": "share_pct",
            "y_field": "price_segment",
            "category_field": "make",
            "value_field": "n_observations",
            "supports_price_segment_filter": True,
            "supports_category_filter": True,
        },
    ]
).sort_values("chart_order").reset_index(drop=True)

# --------------------------------------------
# 1. Προσθήκη metadata για τα plot αρχεία
# --------------------------------------------

plot_key_to_filename = {
    "price_segment_share_bar": required_plot_files["price_segment_share_bar"].name,
    "fuel_type_mix_stacked_bar": required_plot_files["fuel_type_mix_stacked_bar"].name,
    "transmission_mix_stacked_bar": required_plot_files["transmission_mix_stacked_bar"].name,
    "make_mix_stacked_bar": required_plot_files["make_mix_stacked_bar"].name,
}

plot_key_to_path = {
    "price_segment_share_bar": str(required_plot_files["price_segment_share_bar"]),
    "fuel_type_mix_stacked_bar": str(required_plot_files["fuel_type_mix_stacked_bar"]),
    "transmission_mix_stacked_bar": str(required_plot_files["transmission_mix_stacked_bar"]),
    "make_mix_stacked_bar": str(required_plot_files["make_mix_stacked_bar"]),
}

dashboard_charts_registry["plot_file_name"] = dashboard_charts_registry["plot_asset_key"].map(plot_key_to_filename)
dashboard_charts_registry["plot_file_path"] = dashboard_charts_registry["plot_asset_key"].map(plot_key_to_path)

# --------------------------------------------
# 2. Προσθήκη metadata για τα input tables
# --------------------------------------------

table_key_to_filename = {
    "price_segment_share_plot_data": required_csv_files["price_segment_share_plot_data"].name,
    "fuel_type_mix_plot_data": required_csv_files["fuel_type_mix_plot_data"].name,
    "transmission_mix_plot_data": required_csv_files["transmission_mix_plot_data"].name,
    "make_mix_plot_data": required_csv_files["make_mix_plot_data"].name,
}

table_key_to_n_rows = {
    "price_segment_share_plot_data": price_segment_share_plot_data.shape[0],
    "fuel_type_mix_plot_data": fuel_type_mix_plot_data.shape[0],
    "transmission_mix_plot_data": transmission_mix_plot_data.shape[0],
    "make_mix_plot_data": make_mix_plot_data.shape[0],
}

dashboard_charts_registry["data_file_name"] = dashboard_charts_registry["data_table_key"].map(table_key_to_filename)
dashboard_charts_registry["data_n_rows"] = dashboard_charts_registry["data_table_key"].map(table_key_to_n_rows)

# --------------------------------------------
# 3. Γρήγορος έλεγχος αποτελέσματος
# --------------------------------------------

print("Το dashboard chart registry δημιουργήθηκε επιτυχώς.\n")
display(dashboard_charts_registry)

Το dashboard chart registry δημιουργήθηκε επιτυχώς.



,chart_order,chart_id,section_id,chart_title_el,chart_title_en,chart_type,data_table_key,plot_asset_key,x_field,y_field,category_field,value_field,supports_price_segment_filter,supports_category_filter,plot_file_name,plot_file_path,data_file_name,data_n_rows
0,1,chart_price_segment_share,segment_profile,Κατανομή της αγοράς ανά price segment,Market Share by Price Segment,horizontal_bar,price_segment_share_plot_data,price_segment_share_bar,share_pct,price_segment,NaN,n_observations,False,False,notebook6_price_segment_share_bar.png,C:\Users\athin\Desktop\Car-Market-Analysis-Att...,notebook6_price_segment_share_plot_data.csv,4
1,2,chart_fuel_mix,fuel_mix,Σύνθεση τύπου καυσίμου ανά price segment,Fuel Type Mix by Price Segment,stacked_horizontal_bar,fuel_type_mix_plot_data,fuel_type_mix_stacked_bar,share_pct,price_segment,fuel_type,n_observations,True,True,notebook6_fuel_type_mix_stacked_bar.png,C:\Users\athin\Desktop\Car-Market-Analysis-Att...,notebook6_fuel_type_mix_plot_data.csv,36
2,3,chart_transmission_mix,transmission_mix,Σύνθεση τύπου μετάδοσης ανά price segment,Transmission Mix by Price Segment,stacked_horizontal_bar,transmission_mix_plot_data,transmission_mix_stacked_bar,share_pct,price_segment,transmission_type,n_observations,True,True,notebook6_transmission_mix_stacked_bar.png,C:\Users\athin\Desktop\Car-Market-Analysis-Att...,notebook6_transmission_mix_plot_data.csv,12
3,4,chart_make_mix,make_concentration,Σύνθεση βασικών κατασκευαστών ανά price segment,Make Mix by Price Segment,stacked_horizontal_bar,make_mix_plot_data,make_mix_stacked_bar,share_pct,price_segment,make,n_observations,True,True,notebook6_make_mix_stacked_bar.png,C:\Users\athin\Desktop\Car-Market-Analysis-Att...,notebook6_make_mix_plot_data.csv,40


## Streamlit-ready section bundles και component mappings

Στο παρόν στάδιο οργανώνονται οι βασικές ενότητες του dashboard σε μορφή πιο κοντά
στη λογική μίας μελλοντικής εφαρμογής Streamlit.

Στόχος είναι κάθε `section_id` να συνδέεται με:
- το βασικό summary table του,
- τα KPI cards που του αντιστοιχούν,
- τα charts που το υποστηρίζουν,
- και μία συνοπτική περιγραφή των διαθέσιμων components.

Με αυτόν τον τρόπο, το notebook δεν παράγει μόνο registries αποθήκευσης,
αλλά και μία ενιαία layer οργάνωσης που μπορεί να αξιοποιηθεί αργότερα
σε rendering logic, navigation structure και section-based dashboard assembly.

In [11]:
# ============================================
# Notebook 7 - Streamlit-ready section bundles
# ============================================

# --------------------------------------------
# 1. Σύνοψη cards ανά section
# --------------------------------------------

cards_by_section_summary = (
    dashboard_cards_registry
    .groupby("section_id", dropna=False)
    .agg(
        n_cards=("card_id", "count"),
        n_market_cards=("card_scope", lambda s: (s == "market").sum()),
        n_segment_cards=("card_scope", lambda s: (s == "segment").sum()),
    )
    .reset_index()
)

# --------------------------------------------
# 2. Σύνοψη charts ανά section
# --------------------------------------------

charts_by_section_summary = (
    dashboard_charts_registry
    .groupby("section_id", dropna=False)
    .agg(
        n_charts=("chart_id", "count"),
        chart_ids=("chart_id", lambda s: " | ".join(s.astype(str))),
        chart_types=("chart_type", lambda s: " | ".join(s.astype(str))),
    )
    .reset_index()
)

# --------------------------------------------
# 3. Ενοποίηση σε section component registry
# --------------------------------------------

section_component_registry = (
    dashboard_sections_registry
    .merge(cards_by_section_summary, on="section_id", how="left")
    .merge(charts_by_section_summary, on="section_id", how="left")
    .fillna({
        "n_cards": 0,
        "n_market_cards": 0,
        "n_segment_cards": 0,
        "n_charts": 0,
        "chart_ids": "",
        "chart_types": "",
    })
)

integer_like_columns = ["n_cards", "n_market_cards", "n_segment_cards", "n_charts"]
for column_name in integer_like_columns:
    section_component_registry[column_name] = section_component_registry[column_name].astype(int)

# --------------------------------------------
# 4. Δημιουργία αναλυτικών section bundles
# --------------------------------------------

section_bundle_records = []

for _, section_row in dashboard_sections_registry.iterrows():
    section_id = section_row["section_id"]

    section_cards = (
        dashboard_cards_registry.loc[dashboard_cards_registry["section_id"] == section_id, "card_id"]
        .astype("string")
        .tolist()
    )

    section_charts = (
        dashboard_charts_registry.loc[dashboard_charts_registry["section_id"] == section_id, "chart_id"]
        .astype("string")
        .tolist()
    )

    section_bundle_records.append(
        {
            "section_id": section_id,
            "section_order": section_row["section_order"],
            "section_title_el": section_row["section_title_el"],
            "streamlit_tab_name": section_row["streamlit_tab_name"],
            "primary_table_key": section_row["primary_table_key"],
            "primary_plot_key": section_row["primary_plot_key"],
            "default_component_type": section_row["default_component_type"],
            "supports_price_segment_filter": section_row["supports_price_segment_filter"],
            "supports_category_filter": section_row["supports_category_filter"],
            "n_cards": len(section_cards),
            "n_charts": len(section_charts),
            "card_ids": " | ".join(section_cards),
            "chart_ids": " | ".join(section_charts),
        }
    )

streamlit_section_bundles = (
    pd.DataFrame(section_bundle_records)
    .sort_values(["section_order", "section_id"])
    .reset_index(drop=True)
)

# --------------------------------------------
# 5. Validation
# --------------------------------------------

if streamlit_section_bundles["section_id"].duplicated().any():
    raise ValueError("Βρέθηκαν duplicate section_id values στα streamlit section bundles.")

if section_component_registry["section_id"].duplicated().any():
    raise ValueError("Βρέθηκαν duplicate section_id values στο section component registry.")

# --------------------------------------------
# 6. Preview
# --------------------------------------------

print("Το section component registry δημιουργήθηκε επιτυχώς.\n")
display(
    section_component_registry[
        [
            "section_order",
            "section_id",
            "section_title_el",
            "n_cards",
            "n_market_cards",
            "n_segment_cards",
            "n_charts",
            "chart_ids",
        ]
    ]
)

print("\nΤο streamlit-ready section bundles table δημιουργήθηκε επιτυχώς.\n")
display(streamlit_section_bundles)

Το section component registry δημιουργήθηκε επιτυχώς.



,section_order,section_id,section_title_el,n_cards,n_market_cards,n_segment_cards,n_charts,chart_ids
0,1,market_overview,Επισκόπηση αγοράς,11,11,0,0,
1,2,segment_profile,Προφίλ price segments,48,0,48,1,chart_price_segment_share
2,3,fuel_mix,Σύνθεση καυσίμου,0,0,0,1,chart_fuel_mix
3,4,transmission_mix,Σύνθεση μετάδοσης,0,0,0,1,chart_transmission_mix
4,5,make_concentration,Συγκέντρωση κατασκευαστών,0,0,0,1,chart_make_mix



Το streamlit-ready section bundles table δημιουργήθηκε επιτυχώς.



,section_id,section_order,section_title_el,streamlit_tab_name,primary_table_key,primary_plot_key,default_component_type,supports_price_segment_filter,supports_category_filter,n_cards,n_charts,card_ids,chart_ids
0,market_overview,1,Επισκόπηση αγοράς,market_overview,market_overview_kpi_summary,NaN,kpi_cards,False,False,11,0,card_market_market_n_observations | card_marke...,
1,segment_profile,2,Προφίλ price segments,segment_profile,segment_kpi_summary,price_segment_share_bar,kpi_cards_plus_chart,True,False,48,1,card_segment_μεσοyψηλi_segment_Μεσοϋψηλή_n_obs...,chart_price_segment_share
2,fuel_mix,3,Σύνθεση καυσίμου,fuel_mix,fuel_type_dashboard_summary,fuel_type_mix_stacked_bar,stacked_bar_chart,True,True,0,1,,chart_fuel_mix
3,transmission_mix,4,Σύνθεση μετάδοσης,transmission_mix,transmission_dashboard_summary,transmission_mix_stacked_bar,stacked_bar_chart,True,True,0,1,,chart_transmission_mix
4,make_concentration,5,Συγκέντρωση κατασκευαστών,make_concentration,make_dashboard_summary,make_mix_stacked_bar,stacked_bar_chart,True,True,0,1,,chart_make_mix


## Εξαγωγή των registries του Notebook 7

Στο παρόν στάδιο αποθηκεύονται τα βασικά registries που δημιουργήθηκαν στο Notebook 7,
ώστε να υπάρχουν ως καθαρά και επαναχρησιμοποιήσιμα outputs στον φάκελο `data/processed/`.

Η εξαγωγή αυτή είναι σημαντική διότι:
- μετατρέπει τη λογική του notebook σε version-controlled παραδοτέα,
- υποστηρίζει reproducible downstream χρήση,
- και θέτει τη βάση για μελλοντική ενσωμάτωση σε Streamlit app ή σε επόμενο notebook/stage.

In [12]:
# ============================================
# Notebook 7 - Εξαγωγή registries σε CSV
# ============================================

def export_dataframe_csv(df: pd.DataFrame, output_path: Path) -> None:
    """
    Αποθηκεύει DataFrame σε CSV με UTF-8 BOM για ασφαλή συμβατότητα
    με ελληνικά και spreadsheet viewers.
    """
    df.to_csv(output_path, index=False, encoding="utf-8-sig")


# --------------------------------------------
# 1. Ορισμός outputs του Notebook 7
# --------------------------------------------

notebook7_registry_outputs = {
    f"{NOTEBOOK_EXPORT_PREFIX}_dashboard_sections_registry.csv": dashboard_sections_registry,
    f"{NOTEBOOK_EXPORT_PREFIX}_dashboard_cards_registry.csv": dashboard_cards_registry,
    f"{NOTEBOOK_EXPORT_PREFIX}_dashboard_charts_registry.csv": dashboard_charts_registry,
    f"{NOTEBOOK_EXPORT_PREFIX}_section_component_registry.csv": section_component_registry,
    f"{NOTEBOOK_EXPORT_PREFIX}_streamlit_section_bundles.csv": streamlit_section_bundles,
}

# --------------------------------------------
# 2. Εξαγωγή των αρχείων
# --------------------------------------------

export_records = []

for file_name, df in notebook7_registry_outputs.items():
    output_path = PROCESSED_DIR / file_name
    export_dataframe_csv(df, output_path)

    export_records.append(
        {
            "file_name": file_name,
            "output_path": str(output_path),
            "n_rows": df.shape[0],
            "n_columns": df.shape[1],
        }
    )

notebook7_exports_inventory = (
    pd.DataFrame(export_records)
    .sort_values("file_name")
    .reset_index(drop=True)
)

print("Τα βασικά registries του Notebook 7 αποθηκεύτηκαν επιτυχώς.\n")
display(notebook7_exports_inventory)

Τα βασικά registries του Notebook 7 αποθηκεύτηκαν επιτυχώς.



,file_name,output_path,n_rows,n_columns
0,notebook7_dashboard_cards_registry.csv,C:\Users\athin\Desktop\Car-Market-Analysis-Att...,59,14
1,notebook7_dashboard_charts_registry.csv,C:\Users\athin\Desktop\Car-Market-Analysis-Att...,4,18
2,notebook7_dashboard_sections_registry.csv,C:\Users\athin\Desktop\Car-Market-Analysis-Att...,5,11
3,notebook7_section_component_registry.csv,C:\Users\athin\Desktop\Car-Market-Analysis-Att...,5,17
4,notebook7_streamlit_section_bundles.csv,C:\Users\athin\Desktop\Car-Market-Analysis-Att...,5,13


## Τελικός έλεγχος εξαγόμενων παραδοτέων του Notebook 7

Στο παρόν στάδιο πραγματοποιείται ένας τελικός έλεγχος των βασικών outputs
που παρήχθησαν στο Notebook 7.

Ο σκοπός του ελέγχου είναι να επιβεβαιωθεί ότι:
- τα αρχεία αποθηκεύτηκαν επιτυχώς στον φάκελο `data/processed/`,
- μπορούν να αναγνωστούν ξανά χωρίς σφάλμα,
- και το πρώτο reproducible deliverable του παρόντος notebook είναι έτοιμο
  για downstream χρήση και μελλοντική ενσωμάτωση σε Streamlit app.

In [13]:
# ============================================
# Notebook 7 - Τελικός έλεγχος exported outputs
# ============================================

validation_records = []

for file_name in notebook7_registry_outputs.keys():
    file_path = PROCESSED_DIR / file_name

    file_exists = file_path.exists()

    if not file_exists:
        validation_records.append(
            {
                "file_name": file_name,
                "file_exists": False,
                "can_read": False,
                "n_rows": pd.NA,
                "n_columns": pd.NA,
            }
        )
        continue

    try:
        df_check = pd.read_csv(file_path)
        validation_records.append(
            {
                "file_name": file_name,
                "file_exists": True,
                "can_read": True,
                "n_rows": df_check.shape[0],
                "n_columns": df_check.shape[1],
            }
        )
    except Exception:
        validation_records.append(
            {
                "file_name": file_name,
                "file_exists": True,
                "can_read": False,
                "n_rows": pd.NA,
                "n_columns": pd.NA,
            }
        )

notebook7_validation_df = (
    pd.DataFrame(validation_records)
    .sort_values("file_name")
    .reset_index(drop=True)
)

if not notebook7_validation_df["file_exists"].all():
    display(notebook7_validation_df)
    raise FileNotFoundError("Κάποια exported αρχεία του Notebook 7 δεν βρέθηκαν στο data/processed/.")

if not notebook7_validation_df["can_read"].all():
    display(notebook7_validation_df)
    raise ValueError("Κάποια exported αρχεία του Notebook 7 δεν μπορούν να διαβαστούν σωστά.")

print("Ο τελικός έλεγχος των exported outputs ολοκληρώθηκε επιτυχώς.\n")
display(notebook7_validation_df)

print("\nCheckpoint:")
print("- Το πρώτο deliverable του Notebook 7 είναι έτοιμο.")
print("- Τα registries έχουν εξαχθεί επιτυχώς.")
print("- Η δομή για dashboard consumption και Streamlit preparation έχει τεθεί.")

Ο τελικός έλεγχος των exported outputs ολοκληρώθηκε επιτυχώς.



,file_name,file_exists,can_read,n_rows,n_columns
0,notebook7_dashboard_cards_registry.csv,True,True,59,14
1,notebook7_dashboard_charts_registry.csv,True,True,4,18
2,notebook7_dashboard_sections_registry.csv,True,True,5,11
3,notebook7_section_component_registry.csv,True,True,5,17
4,notebook7_streamlit_section_bundles.csv,True,True,5,13



Checkpoint:
- Το πρώτο deliverable του Notebook 7 είναι έτοιμο.
- Τα registries έχουν εξαχθεί επιτυχώς.
- Η δομή για dashboard consumption και Streamlit preparation έχει τεθεί.


## Refinement layer για streamlit-friendly filters

Στο παρόν στάδιο δημιουργούνται οι βασικές επιλογές φίλτρων που θα μπορούσαν
να χρησιμοποιηθούν σε μελλοντική εφαρμογή Streamlit.

Ο στόχος είναι να οριστούν με καθαρό και αναπαραγώγιμο τρόπο:
- οι διαθέσιμες επιλογές για `price_segment`,
- οι επιλογές για `fuel_type`,
- οι επιλογές για `transmission_type`,
- και οι επιλογές για `make`.

Με αυτόν τον τρόπο, τα dashboard inputs δεν παραμένουν μόνο σε επίπεδο tables
και registries, αλλά επεκτείνονται και σε μία πρώτη layer αλληλεπίδρασης,
χρήσιμη για φίλτρα, selectboxes και multiselect components.

In [14]:
# ============================================
# Notebook 7 - Streamlit-friendly filter options registry
# ============================================

def build_filter_options_registry() -> pd.DataFrame:
    """
    Δημιουργεί registry με διαθέσιμες επιλογές φίλτρων για μελλοντική χρήση
    σε Streamlit components.
    """
    records = []

    # --------------------------------------------
    # 1. Price segment options
    # --------------------------------------------

    available_price_segments = [
        segment
        for segment in canonical_segment_order
        if segment in price_segment_share_plot_data["price_segment"].astype("string").tolist()
    ]

    records.append(
        {
            "filter_group": "price_segment",
            "option_order": 0,
            "option_value": "ALL",
            "option_label_el": "Όλα τα segments",
            "option_label_en": "All Segments",
            "section_ids_supported": "segment_profile | fuel_mix | transmission_mix | make_concentration",
            "source_table_key": "price_segment_share_plot_data",
            "is_default_option": True,
        }
    )

    for idx, segment_value in enumerate(available_price_segments, start=1):
        records.append(
            {
                "filter_group": "price_segment",
                "option_order": idx,
                "option_value": str(segment_value),
                "option_label_el": str(segment_value),
                "option_label_en": str(segment_value),
                "section_ids_supported": "segment_profile | fuel_mix | transmission_mix | make_concentration",
                "source_table_key": "price_segment_share_plot_data",
                "is_default_option": False,
            }
        )

    # --------------------------------------------
    # 2. Fuel type options
    # --------------------------------------------

    fuel_type_values = sorted(
        fuel_type_dashboard_summary["fuel_type"]
        .astype("string")
        .dropna()
        .unique()
        .tolist()
    )

    records.append(
        {
            "filter_group": "fuel_type",
            "option_order": 0,
            "option_value": "ALL",
            "option_label_el": "Όλοι οι τύποι καυσίμου",
            "option_label_en": "All Fuel Types",
            "section_ids_supported": "fuel_mix",
            "source_table_key": "fuel_type_dashboard_summary",
            "is_default_option": True,
        }
    )

    for idx, fuel_value in enumerate(fuel_type_values, start=1):
        records.append(
            {
                "filter_group": "fuel_type",
                "option_order": idx,
                "option_value": str(fuel_value),
                "option_label_el": str(fuel_value),
                "option_label_en": str(fuel_value),
                "section_ids_supported": "fuel_mix",
                "source_table_key": "fuel_type_dashboard_summary",
                "is_default_option": False,
            }
        )

    # --------------------------------------------
    # 3. Transmission type options
    # --------------------------------------------

    transmission_values = sorted(
        transmission_dashboard_summary["transmission_type"]
        .astype("string")
        .dropna()
        .unique()
        .tolist()
    )

    records.append(
        {
            "filter_group": "transmission_type",
            "option_order": 0,
            "option_value": "ALL",
            "option_label_el": "Όλοι οι τύποι μετάδοσης",
            "option_label_en": "All Transmission Types",
            "section_ids_supported": "transmission_mix",
            "source_table_key": "transmission_dashboard_summary",
            "is_default_option": True,
        }
    )

    for idx, transmission_value in enumerate(transmission_values, start=1):
        records.append(
            {
                "filter_group": "transmission_type",
                "option_order": idx,
                "option_value": str(transmission_value),
                "option_label_el": str(transmission_value),
                "option_label_en": str(transmission_value),
                "section_ids_supported": "transmission_mix",
                "source_table_key": "transmission_dashboard_summary",
                "is_default_option": False,
            }
        )

    # --------------------------------------------
    # 4. Make options
    # --------------------------------------------

    make_values = sorted(
        make_dashboard_summary["make"]
        .astype("string")
        .dropna()
        .unique()
        .tolist()
    )

    records.append(
        {
            "filter_group": "make",
            "option_order": 0,
            "option_value": "ALL",
            "option_label_el": "Όλοι οι κατασκευαστές",
            "option_label_en": "All Makes",
            "section_ids_supported": "make_concentration",
            "source_table_key": "make_dashboard_summary",
            "is_default_option": True,
        }
    )

    for idx, make_value in enumerate(make_values, start=1):
        records.append(
            {
                "filter_group": "make",
                "option_order": idx,
                "option_value": str(make_value),
                "option_label_el": str(make_value),
                "option_label_en": str(make_value),
                "section_ids_supported": "make_concentration",
                "source_table_key": "make_dashboard_summary",
                "is_default_option": False,
            }
        )

    filter_options_registry = (
        pd.DataFrame(records)
        .sort_values(["filter_group", "option_order", "option_value"])
        .reset_index(drop=True)
    )

    return filter_options_registry


dashboard_filter_options_registry = build_filter_options_registry()

print("Το dashboard filter options registry δημιουργήθηκε επιτυχώς.\n")
display(dashboard_filter_options_registry.head(20))

# --------------------------------------------
# 5. Export
# --------------------------------------------

filter_registry_file_name = f"{NOTEBOOK_EXPORT_PREFIX}_dashboard_filter_options_registry.csv"
filter_registry_output_path = PROCESSED_DIR / filter_registry_file_name

export_dataframe_csv(
    dashboard_filter_options_registry,
    filter_registry_output_path,
)

# Κρατάμε το output και στο dictionary των Notebook 7 exports
notebook7_registry_outputs[filter_registry_file_name] = dashboard_filter_options_registry

print(f"\nΤο αρχείο αποθηκεύτηκε στο: {filter_registry_output_path}")

Το dashboard filter options registry δημιουργήθηκε επιτυχώς.



,filter_group,option_order,option_value,option_label_el,option_label_en,section_ids_supported,source_table_key,is_default_option
0,fuel_type,0,ALL,Όλοι οι τύποι καυσίμου,All Fuel Types,fuel_mix,fuel_type_dashboard_summary,True
1,fuel_type,1,CNG / Βενζίνη,CNG / Βενζίνη,CNG / Βενζίνη,fuel_mix,fuel_type_dashboard_summary,False
2,fuel_type,2,LPG / Βενζίνη,LPG / Βενζίνη,LPG / Βενζίνη,fuel_mix,fuel_type_dashboard_summary,False
3,fuel_type,3,Plug-in Hybrid Βενζίνης,Plug-in Hybrid Βενζίνης,Plug-in Hybrid Βενζίνης,fuel_mix,fuel_type_dashboard_summary,False
4,fuel_type,4,Plug-in Hybrid Πετρελαίου,Plug-in Hybrid Πετρελαίου,Plug-in Hybrid Πετρελαίου,fuel_mix,fuel_type_dashboard_summary,False
5,fuel_type,5,Βενζίνη,Βενζίνη,Βενζίνη,fuel_mix,fuel_type_dashboard_summary,False
6,fuel_type,6,Ηλεκτρικό,Ηλεκτρικό,Ηλεκτρικό,fuel_mix,fuel_type_dashboard_summary,False
7,fuel_type,7,Πετρέλαιο,Πετρέλαιο,Πετρέλαιο,fuel_mix,fuel_type_dashboard_summary,False
8,fuel_type,8,Υβριδικό Βενζίνης,Υβριδικό Βενζίνης,Υβριδικό Βενζίνης,fuel_mix,fuel_type_dashboard_summary,False
9,fuel_type,9,Υβριδικό Πετρελαίου,Υβριδικό Πετρελαίου,Υβριδικό Πετρελαίου,fuel_mix,fuel_type_dashboard_summary,False



Το αρχείο αποθηκεύτηκε στο: C:\Users\athin\Desktop\Car-Market-Analysis-Attica\data\processed\notebook7_dashboard_filter_options_registry.csv


## Integration asset inventory για downstream χρήση

Στο παρόν στάδιο δημιουργείται ένας συνολικός inventory πίνακας των βασικών assets
που σχετίζονται με το Notebook 7.

Ο inventory αυτός περιλαμβάνει:
- τα βασικά input assets που καταναλώθηκαν από το Notebook 6,
- και τα νέα registries που δημιουργήθηκαν στο Notebook 7.

Η ύπαρξη ενός τέτοιου inventory διευκολύνει:
- την τεκμηρίωση του pipeline,
- την downstream ενσωμάτωση σε Streamlit app,
- και τον τελικό έλεγχο της λογικής εξαρτήσεων μεταξύ inputs και outputs.

In [15]:
# ============================================
# Notebook 7 - Integration asset inventory
# ============================================

# --------------------------------------------
# 1. Inventory των inputs από το Notebook 6
# --------------------------------------------

notebook6_input_assets_inventory = loaded_inputs_inventory.copy()
notebook6_input_assets_inventory["asset_stage"] = "input_from_notebook6"

notebook6_input_assets_inventory = notebook6_input_assets_inventory.rename(
    columns={
        "asset_key": "asset_id",
        "asset_kind": "asset_kind",
        "file_name": "file_name",
    }
)

notebook6_input_assets_inventory["asset_role"] = notebook6_input_assets_inventory["asset_id"].map(
    {
        "market_overview_kpi_summary": "market_kpi_table",
        "segment_kpi_summary": "segment_kpi_table",
        "fuel_type_dashboard_summary": "fuel_summary_table",
        "transmission_dashboard_summary": "transmission_summary_table",
        "make_dashboard_summary": "make_summary_table",
        "price_segment_share_plot_data": "plot_data_table",
        "fuel_type_mix_plot_data": "plot_data_table",
        "transmission_mix_plot_data": "plot_data_table",
        "make_mix_plot_data": "plot_data_table",
        "top10_make_by_segment_counts_corrected": "corrected_make_counts_table",
        "top10_make_by_segment_shares_corrected": "corrected_make_shares_table",
        "price_segment_share_bar": "plot_asset",
        "fuel_type_mix_stacked_bar": "plot_asset",
        "transmission_mix_stacked_bar": "plot_asset",
        "make_mix_stacked_bar": "plot_asset",
    }
)

# --------------------------------------------
# 2. Inventory των outputs του Notebook 7
# --------------------------------------------

notebook7_output_records = []

for file_name, df in notebook7_registry_outputs.items():
    notebook7_output_records.append(
        {
            "asset_stage": "generated_in_notebook7",
            "asset_id": file_name.replace(".csv", ""),
            "asset_kind": "csv_registry",
            "file_name": file_name,
            "asset_role": file_name.replace(f"{NOTEBOOK_EXPORT_PREFIX}_", "").replace(".csv", ""),
            "n_rows": df.shape[0],
            "n_columns": df.shape[1],
        }
    )

notebook7_output_assets_inventory = pd.DataFrame(notebook7_output_records)

# --------------------------------------------
# 3. Ενοποίηση inventories
# --------------------------------------------

common_columns = [
    "asset_stage",
    "asset_id",
    "asset_kind",
    "file_name",
    "asset_role",
    "n_rows",
    "n_columns",
]

notebook6_input_assets_inventory = notebook6_input_assets_inventory[common_columns].copy()
notebook7_output_assets_inventory = notebook7_output_assets_inventory[common_columns].copy()

notebook7_integration_asset_inventory = (
    pd.concat(
        [notebook6_input_assets_inventory, notebook7_output_assets_inventory],
        axis=0,
        ignore_index=True,
    )
    .sort_values(["asset_stage", "asset_kind", "file_name"])
    .reset_index(drop=True)
)

print("Το integration asset inventory δημιουργήθηκε επιτυχώς.\n")
display(notebook7_integration_asset_inventory)

# --------------------------------------------
# 4. Export
# --------------------------------------------

integration_inventory_file_name = f"{NOTEBOOK_EXPORT_PREFIX}_integration_asset_inventory.csv"
integration_inventory_output_path = PROCESSED_DIR / integration_inventory_file_name

export_dataframe_csv(
    notebook7_integration_asset_inventory,
    integration_inventory_output_path,
)

notebook7_registry_outputs[integration_inventory_file_name] = notebook7_integration_asset_inventory

print(f"\nΤο αρχείο αποθηκεύτηκε στο: {integration_inventory_output_path}")

Το integration asset inventory δημιουργήθηκε επιτυχώς.



,asset_stage,asset_id,asset_kind,file_name,asset_role,n_rows,n_columns
0,generated_in_notebook7,notebook7_dashboard_cards_registry,csv_registry,notebook7_dashboard_cards_registry.csv,dashboard_cards_registry,59.00,14.00
1,generated_in_notebook7,notebook7_dashboard_charts_registry,csv_registry,notebook7_dashboard_charts_registry.csv,dashboard_charts_registry,4.00,18.00
2,generated_in_notebook7,notebook7_dashboard_filter_options_registry,csv_registry,notebook7_dashboard_filter_options_registry.csv,dashboard_filter_options_registry,30.00,8.00
3,generated_in_notebook7,notebook7_dashboard_sections_registry,csv_registry,notebook7_dashboard_sections_registry.csv,dashboard_sections_registry,5.00,11.00
4,generated_in_notebook7,notebook7_section_component_registry,csv_registry,notebook7_section_component_registry.csv,section_component_registry,5.00,17.00
5,generated_in_notebook7,notebook7_streamlit_section_bundles,csv_registry,notebook7_streamlit_section_bundles.csv,streamlit_section_bundles,5.00,13.00
6,input_from_notebook6,fuel_type_dashboard_summary,csv_table,notebook6_fuel_type_dashboard_summary.csv,fuel_summary_table,36.00,8.00
7,input_from_notebook6,fuel_type_mix_plot_data,csv_table,notebook6_fuel_type_mix_plot_data.csv,plot_data_table,36.00,8.00
8,input_from_notebook6,make_dashboard_summary,csv_table,notebook6_make_dashboard_summary.csv,make_summary_table,40.00,8.00
9,input_from_notebook6,make_mix_plot_data,csv_table,notebook6_make_mix_plot_data.csv,plot_data_table,40.00,8.00



Το αρχείο αποθηκεύτηκε στο: C:\Users\athin\Desktop\Car-Market-Analysis-Attica\data\processed\notebook7_integration_asset_inventory.csv


## Τελικός επανέλεγχος όλων των outputs του Notebook 7

Μετά την προσθήκη του refinement layer για streamlit-friendly filters και του
integration asset inventory, πραγματοποιείται ένας τελικός επανέλεγχος όλων
των εξαγόμενων outputs του Notebook 7.

Στόχος είναι να επιβεβαιωθεί ότι:
- όλα τα τελικά CSV αρχεία υπάρχουν στον φάκελο `data/processed/`,
- όλα μπορούν να αναγνωστούν ξανά επιτυχώς,
- και το notebook έχει πλέον παραγάγει ένα πλήρες και συνεπές σύνολο
  downstream-ready assets για dashboard consumption και μελλοντικό Streamlit integration.

In [16]:
# ============================================
# Notebook 7 - Τελικός επανέλεγχος όλων των outputs
# ============================================

final_validation_records = []

for file_name in sorted(notebook7_registry_outputs.keys()):
    file_path = PROCESSED_DIR / file_name
    file_exists = file_path.exists()

    if not file_exists:
        final_validation_records.append(
            {
                "file_name": file_name,
                "file_exists": False,
                "can_read": False,
                "n_rows": pd.NA,
                "n_columns": pd.NA,
            }
        )
        continue

    try:
        df_check = pd.read_csv(file_path)
        final_validation_records.append(
            {
                "file_name": file_name,
                "file_exists": True,
                "can_read": True,
                "n_rows": df_check.shape[0],
                "n_columns": df_check.shape[1],
            }
        )
    except Exception:
        final_validation_records.append(
            {
                "file_name": file_name,
                "file_exists": True,
                "can_read": False,
                "n_rows": pd.NA,
                "n_columns": pd.NA,
            }
        )

notebook7_final_validation_df = (
    pd.DataFrame(final_validation_records)
    .sort_values("file_name")
    .reset_index(drop=True)
)

if not notebook7_final_validation_df["file_exists"].all():
    display(notebook7_final_validation_df)
    raise FileNotFoundError(
        "Κάποια τελικά exported αρχεία του Notebook 7 δεν βρέθηκαν στο data/processed/."
    )

if not notebook7_final_validation_df["can_read"].all():
    display(notebook7_final_validation_df)
    raise ValueError(
        "Κάποια τελικά exported αρχεία του Notebook 7 δεν μπορούν να διαβαστούν σωστά."
    )

print("Ο τελικός επανέλεγχος όλων των outputs του Notebook 7 ολοκληρώθηκε επιτυχώς.\n")
display(notebook7_final_validation_df)

print("\nΣύνοψη τελικών παραδοτέων:")
print(f"- Συνολικά exported files: {len(notebook7_final_validation_df)}")
print(f"- Όλα τα αρχεία υπάρχουν: {notebook7_final_validation_df['file_exists'].all()}")
print(f"- Όλα τα αρχεία διαβάζονται σωστά: {notebook7_final_validation_df['can_read'].all()}")

Ο τελικός επανέλεγχος όλων των outputs του Notebook 7 ολοκληρώθηκε επιτυχώς.



,file_name,file_exists,can_read,n_rows,n_columns
0,notebook7_dashboard_cards_registry.csv,True,True,59,14
1,notebook7_dashboard_charts_registry.csv,True,True,4,18
2,notebook7_dashboard_filter_options_registry.csv,True,True,30,8
3,notebook7_dashboard_sections_registry.csv,True,True,5,11
4,notebook7_integration_asset_inventory.csv,True,True,21,7
5,notebook7_section_component_registry.csv,True,True,5,17
6,notebook7_streamlit_section_bundles.csv,True,True,5,13



Σύνοψη τελικών παραδοτέων:
- Συνολικά exported files: 7
- Όλα τα αρχεία υπάρχουν: True
- Όλα τα αρχεία διαβάζονται σωστά: True


## Συνοπτική αποτίμηση του Notebook 7

Το `07_Dashboard_Consumption_Streamlit_Prep.ipynb` ολοκλήρωσε με επιτυχία
τη μετάβαση από τα dashboard-ready outputs του Notebook 6 σε μία πιο ώριμη
και δομημένη layer κατάλληλη για downstream dashboard consumption.

Συγκεκριμένα, στο παρόν notebook πραγματοποιήθηκαν:

- η φόρτωση και το validation των βασικών CSV summaries και plot assets,
- η τυποποίηση και σταθερή ταξινόμηση των dashboard inputs,
- ο ορισμός των βασικών dashboard sections / views,
- η δημιουργία registry για KPI cards,
- η δημιουργία registry για charts,
- η δημιουργία streamlit-ready section bundles,
- η δημιουργία filter options registry,
- και η σύνθεση ενός integration asset inventory για downstream χρήση.

Με βάση τα παραπάνω, το project διαθέτει πλέον μία πρώτη ολοκληρωμένη
δομή orchestration για dashboard κατανάλωση, η οποία μπορεί να αξιοποιηθεί
είτε σε επόμενο notebook/stage είτε σε μελλοντική εφαρμογή Streamlit.